# Анализ результатов сценариев и ИПУ

Жизненный цикл ФП/СФП:

- **Вариант 1:** Возникновение ФП/СФП → Сценарий → Результат сценария → Закрытие
- **Вариант 2:** Возникновение ФП/СФП → Сценарий → Негативный результат → ИПУ → Результат ИПУ → Закрытие

Дедупликация: если нет дат ИПУ — берём запись с самой поздней датой сценария;  
если есть даты ИПУ — берём запись с самой поздней датой ИПУ.

## 0. Конфигурация, загрузка, нормализация, классификация

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]
DEDUP_KEY = ["inn", "fp_num", "fp_type", "IDENTIFICATION_DATE"]

SCR_DATE_COLS = ["END_DATE_SCR_FCT", "END_DATE_SCR_PLAN"]
IPU_DATE_COLS = ["FIRST_END_DATE_EVENT", "NEW_PLAN_END_DATE_EVT", "END_EVENT_DATE_FACT"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


print("OK")

In [ ]:
raw = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
raw.columns = raw.columns.str.strip()

if "DESC_TEXT.1" in raw.columns and "DESC_TEXT_1" not in raw.columns:
    raw = raw.rename(columns={"DESC_TEXT.1": "DESC_TEXT_1"})
    print("Переименована колонка DESC_TEXT.1 → DESC_TEXT_1")

print(f"Загружено: {len(raw):,} строк")

if "VAL" in raw.columns:
    raw = raw[raw["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"После фильтра по источникам: {len(raw):,}")

raw["inn"] = raw["X_INN"].apply(normalize_inn)
raw["dt"] = pd.to_datetime(raw["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
raw = raw[(raw["dt"] >= DATE_FROM) & (raw["dt"] <= DATE_TO)].copy()
print(f"После фильтра по периоду: {len(raw):,}")

raw["segment"] = raw["X_AREA_RESP"].str.strip().map(SEGMENT_MAP)
raw = raw[raw["segment"].notna()].copy()

before_drop = len(raw)
raw = raw.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(raw):,} строк (удалено {before_drop - len(raw):,})")

raw["fp_num"]  = raw["NUMBER_FP_SFP"].astype(str).str.strip()
raw["fp_type"] = raw["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})
print(f"TYPE_FP значения: {raw['fp_type'].value_counts().to_dict()}")

_LAT2CYR = str.maketrans({
    'A': 'А', 'B': 'В', 'C': 'С', 'E': 'Е', 'H': 'Н', 'K': 'К',
    'M': 'М', 'O': 'О', 'P': 'Р', 'T': 'Т', 'X': 'Х',
    'a': 'а', 'c': 'с', 'e': 'е', 'o': 'о', 'p': 'р', 'x': 'х', 'y': 'у',
})

def norm_text(s):
    if pd.isna(s) or s == "":
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.translate(_LAT2CYR)
    s = re.sub(r'[\s\xa0\u200b\ufeff]+', ' ', s)
    return s.strip()

raw["scenario"] = raw["DESC_TEXT_1"].apply(norm_text)
raw["ipu"]      = raw["DESC_TEXT"].apply(norm_text)

for col in SCR_DATE_COLS + IPU_DATE_COLS + ["DATE_END_FP_SFP"]:
    raw[f"_{col}_dt"] = pd.to_datetime(raw[col], dayfirst=True, errors="coerce")

print(f"Итого: {len(raw):,} строк, {raw['inn'].nunique():,} ИНН")

In [ ]:
SCR_POSITIVE = [
    "ФП/СФП устранен", "ФП/СФП урегулирован", "Микро_У ФП урегулирован",
    "1_ККБ/МСБ_ФП устранен", "СФП устранен",
    "Микро_У Фактор устранен",
    "ФП/СФП устранен (договор закрыт)",
    "2_ККБ/МСБ_СФП устранен",
    "ДКБ_Принято решение УОБ об урегулировании ФП",
    "ФП устранен (договор закрыт)",
    "ДКБ_Не требуется решение УЛ/УОБ о снятии ФП с контроля",
    "Микро_У ФП устранен", "ФП устранен",
    "Микро_У СФП устранен",
    "ДКБ_СФП устранен",
    "Микро_У СФП урегулирован",
]

SCR_COND_POSITIVE = [
    "МСБ_Принято решение УОБ о снятии ФП с контроля",
    "МСБ_Принято решение УЛ о снятии ФП с контроля",
    "Микро_У Требуется принятие решения УЛ о снятии фактора с контроля",
    "Принято решение УЛ о снятии ФП с контроля",
    "ДКБ_Принято решение УЛ о снятии ФП с контроля",
    "Принято решение УОБ о снятии ФП с контроля",
    "ДКБ_Принято решение УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии фактора с контроля",
    "ДКБ_ФП не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, возможно снятие ФП с контроля",
    "ФП урегулирован, требуется принятие решения УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УОБ о снятии ФП с контроля",
    "Микро_У Принято решение УЛ о снятии ФП с контроля",
    "ФП не устранен, не оказывает негативного влияния, требует рассмотрения вопроса на УОБ с целью снятия ФП с контроля",
    "11_ККБ/МСБ_ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_МСБ_Право Банка реализовано, ФП снят с контроля",
    "5_МСБ_Право Банка не реализовано, ФП снят с контроля",
    "Микро_У ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "5_ККБ/МСБ_ФП не требует устранения, право Банка не реализовано",
    "ФП не устранен, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком",
    "ФП не устранен в рамках сценария, не оказывает негативного влияния на исполнение участником кредитной сделки обязательств перед Банком, (требуется решение УОБ о снятии ФП с контроля)",
    "4_ККБ/МСБ_ФП не требует устранения, право Банка реализовано",
    "ФП не требует устранения, право Банка не реализовано",
    "Микро_У Фактор устранен - решение УОБ о снятии фактора с контроля",
    "Микро_У Принятие решения УЛ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ о снятии фактора с контроля",
    "Микро_У Требуется принятие решения УОБ/УЛ о снятии ФП с контроля",
]

SCR_NEGATIVE = [
    "ФП/СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "3_ККБ/МСБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У Фактор не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ДКБ_СФП не устранен, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "12_ККБ/МСБ_ФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "Микро_У СФП не устранен, оказывает негативное влияние, требует рассмотрения на УОБ вопроса о признании задолженности проблемной",
    "ДКБ_ФП оказывает негативное влияние, требует рассмотрения УОБ вопроса о признании задолженности проблемной",
]

SCR_COND_NEGATIVE = [
    "ФП оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ПУ",
    "ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "Микро_У ФП не устранен в рамках сценария, требуется разработка ИПУ",
    "10_ККБ/МСБ_ФП не устранен в рамках сценария, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком, требуется разработка ИПУ",
    "ФП не устранен в рамках сценария, оказывает негативное влияние, требует рассмотрения вопроса на УОБ об утверждении Плана устранения ФП",
    "ФП не устранен. Лимит снижен до \"0\". Лимит восстановлению не подлежит.",
    "Микро_У Техническое закрытие ФП в связи с запуском нового сценария по СФП (СФП 15.2У/15.2.1У)",
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
]

SCR_ERROR = [
    "0_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "ДКБ_Информация проверена, отсутствуют основания для выявления ФП/СФП",
    "Микро_У Информация проверена, отсутствуют основания для выявления ФП/СФП",
]

SCR_ACTIVE = [
    "Активный (срок окончания, предусмотренный Порядком не наступил, находится в стадии реализации)",
    "Активный (срок окончания, предусмотренный Порядком наступил, результат реализации по состоянию на дату Реестра отсутствует)",
]

IPU_POSITIVE = [
    "ФП устранен",
    "ФП устранен (договор закрыт)",
    "Принято решение УОБ о снятии с контроля",
]
IPU_NEGATIVE = [
    "ФП не устранен, выявлены предпосылки, требующие рассмотрения вопроса о признании задолженности проблемной",
]
IPU_ACTIVE = [
    "Активный (продлен первоначальный срок окончания Плана, находится в стадии реализации)",
    "Активный (первоначально установленный срок окончания плана не наступил, находится в стадии реализации)",
    "ФП не устранен в рамках Плана, оказывает негативное влияние на исполнение участником кредитной сделки обязательств перед Банком (требует корректировки ИПУ)",
]
IPU_ERROR = [
    "Активный (срок окончания Плана истек, результат реализации по состоянию на дату Реестра отсутствует)",
]

def norm_list(lst):
    return [norm_text(v) for v in lst]

SCR_POSITIVE      = norm_list(SCR_POSITIVE)
SCR_COND_POSITIVE = norm_list(SCR_COND_POSITIVE)
SCR_NEGATIVE      = norm_list(SCR_NEGATIVE)
SCR_COND_NEGATIVE = norm_list(SCR_COND_NEGATIVE)
SCR_ERROR         = norm_list(SCR_ERROR)
SCR_ACTIVE        = norm_list(SCR_ACTIVE)
IPU_POSITIVE = norm_list(IPU_POSITIVE)
IPU_NEGATIVE = norm_list(IPU_NEGATIVE)
IPU_ACTIVE   = norm_list(IPU_ACTIVE)
IPU_ERROR    = norm_list(IPU_ERROR)

_scr_sets = {
    "положительный":          set(SCR_POSITIVE),
    "условно положительный":  set(SCR_COND_POSITIVE),
    "отрицательный":          set(SCR_NEGATIVE),
    "условно отрицательный":  set(SCR_COND_NEGATIVE),
    "ошибка":                 set(SCR_ERROR),
    "активный":               set(SCR_ACTIVE),
}

_ipu_sets = {
    "положительный": set(IPU_POSITIVE),
    "отрицательный": set(IPU_NEGATIVE),
    "активный":      set(IPU_ACTIVE),
    "ошибка":        set(IPU_ERROR),
}


def classify_scr(val):
    for cls, s in _scr_sets.items():
        if val in s:
            return cls
    return "[пусто]" if val == "" else "неклассифицированный"


def classify_ipu(val):
    for cls, s in _ipu_sets.items():
        if val in s:
            return cls
    return "[пусто]" if val == "" else "неклассифицированный"




print(f"Сценарий: {len(SCR_POSITIVE)} полож., {len(SCR_COND_POSITIVE)} усл.полож., "
      f"{len(SCR_NEGATIVE)} отриц., {len(SCR_COND_NEGATIVE)} усл.отриц., "
      f"{len(SCR_ERROR)} ошибок, {len(SCR_ACTIVE)} активн.")
print(f"ИПУ: {len(IPU_POSITIVE)} полож., {len(IPU_NEGATIVE)} отриц., "
      f"{len(IPU_ACTIVE)} актив., {len(IPU_ERROR)} ошибок")


## 1. Финализированная дедупликация (два потока) — факт-приоритет

- **Поток 1 (только сценарий):** нет ни одной заполненной даты ИПУ → берём запись с приоритетом `END_DATE_SCR_FCT → END_DATE_SCR_PLAN` (сначала фактическая дата, при отсутствии — плановая)
- **Поток 2 (сценарий + ИПУ):** есть хотя бы одна дата ИПУ → берём запись с приоритетом `END_EVENT_DATE_FACT → NEW_PLAN_END_DATE_EVT → FIRST_END_DATE_EVENT`
- Карточки без каких-либо дат — исключаем
- Обоснование: фактическая дата отражает реальное завершение работы по ФП, поэтому имеет приоритет над плановой

In [ ]:
scr_dt = [f"_{c}_dt" for c in SCR_DATE_COLS]
ipu_dt = [f"_{c}_dt" for c in IPU_DATE_COLS]

raw["_scr_max"] = raw[scr_dt].max(axis=1)
raw["_ipu_max"] = raw[ipu_dt].max(axis=1)

# Для каждой группы: есть ли хоть одна заполненная дата ИПУ
grp_has_ipu = raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()

stream1_raw = raw[~grp_has_ipu].copy()
stream2_raw = raw[grp_has_ipu].copy()

# ── Поток 1: факт-приоритет (факт → план) ──
stream1_raw = stream1_raw[
    stream1_raw.groupby(DEDUP_KEY)["_scr_max"].transform("max").notna()
]
stream1_sorted = stream1_raw.sort_values(
    ["_END_DATE_SCR_FCT_dt", "_END_DATE_SCR_PLAN_dt"],
    ascending=[False, False], na_position="last"
)
df_stream1 = stream1_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream1["stream"] = "scenario"

# ── Поток 2: факт-приоритет (факт → новый план → план) ──
stream2_raw = stream2_raw[
    stream2_raw.groupby(DEDUP_KEY)["_ipu_max"].transform("max").notna()
]
stream2_sorted = stream2_raw.sort_values(
    ["_END_EVENT_DATE_FACT_dt", "_NEW_PLAN_END_DATE_EVT_dt", "_FIRST_END_DATE_EVENT_dt"],
    ascending=[False, False, False], na_position="last"
)
df_stream2 = stream2_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first").copy()
df_stream2["stream"] = "ipu"

# «Истинный» результат сценария для Потока 2:
# дедупликация по дате СЦЕНАРИЯ (факт → план)
s2_scr_sorted = stream2_raw.sort_values(
    ["_END_DATE_SCR_FCT_dt", "_END_DATE_SCR_PLAN_dt"],
    ascending=[False, False], na_position="last"
)
s2_scr_dedup = s2_scr_sorted.drop_duplicates(subset=DEDUP_KEY, keep="first")
_true_map = s2_scr_dedup.set_index(DEDUP_KEY)["scenario"].apply(classify_scr)
_true_map.name = "true_scr_class"
df_stream2 = df_stream2.set_index(DEDUP_KEY).join(_true_map).reset_index()
df_stream2["true_scr_class"] = df_stream2["true_scr_class"].fillna("[пусто]")

df = pd.concat([df_stream1, df_stream2], ignore_index=True)

# Классификация
df["scr_class"] = df["scenario"].apply(classify_scr)
df["ipu_class"] = df["ipu"].apply(classify_ipu)
if "true_scr_class" not in df.columns:
    df["true_scr_class"] = df["scr_class"]

total_raw_cards = raw.drop_duplicates(DEDUP_KEY).shape[0]
n1 = len(df_stream1)
n2 = len(df_stream2)
excluded_no_dates = total_raw_cards - n1 - n2

print(f"Всего уникальных карточек ФП (до фильтрации): {total_raw_cards:,}")
print(f"Поток 1 (только сценарий):  {n1:,} ({n1/total_raw_cards*100:.1f}%)")
print(f"Поток 2 (сценарий + ИПУ):   {n2:,} ({n2/total_raw_cards*100:.1f}%)")
print(f"Исключено (нет дат):         {excluded_no_dates:,} ({excluded_no_dates/total_raw_cards*100:.1f}%)")

# Исключаем карточки с пустым результатом сценария
empty_scr = (df["scr_class"] == "[пусто]").sum()
df = df[df["scr_class"] != "[пусто]"].copy()
print(f"Исключено (пустой результат сценария): {empty_scr:,}")

# Исключаем неклассифицированные результаты
uncl_scr = (df["scr_class"] == "неклассифицированный").sum()
df = df[df["scr_class"] != "неклассифицированный"].copy()
print(f"Исключено (неклассифицированный результат): {uncl_scr:,}")

print(f"\nИтого в датасете df:        {len(df):,}")

## 2. Распределение результатов сценариев

In [ ]:
print("=" * 70)
print("  РЕЗУЛЬТАТЫ СЦЕНАРИЕВ (все карточки)")
print("=" * 70)

n_total = len(df)

# Сводка по классам
scr_cls = df["scr_class"].value_counts()
scr_cls_pct = (scr_cls / n_total * 100).round(1)
scr_summary = pd.DataFrame({"Количество": scr_cls, "%": scr_cls_pct})
scr_summary.index.name = "Класс результата"
print(f"\nВсего карточек: {n_total:,}\n")
display(scr_summary)

# Отдельная таблица для каждой группы
for cls_name in ["положительный", "условно положительный", "отрицательный", "условно отрицательный", "ошибка", "активный"]:
    sub = df[df["scr_class"] == cls_name]
    if len(sub) == 0:
        continue
    detail = sub["scenario"].value_counts().reset_index()
    detail.columns = ["Результат сценария", "Количество"]
    detail["% от группы"] = (detail["Количество"] / len(sub) * 100).round(1)
    detail["% от всех"] = (detail["Количество"] / n_total * 100).round(2)
    print(f"\n{'─'*60}")
    print(f"  {cls_name.upper()} ({len(sub):,} карточек, {len(sub)/n_total*100:.1f}%)")
    print(f"{'─'*60}")
    display(detail.style.hide(axis="index"))

## 3. Распределение результатов ИПУ

In [ ]:
df_ipu_all = df[df["stream"] == "ipu"].copy()
empty_ipu = (df_ipu_all["ipu_class"] == "[пусто]").sum()
uncl_ipu = (df_ipu_all["ipu_class"] == "неклассифицированный").sum()
df_ipu = df_ipu_all[~df_ipu_all["ipu_class"].isin(["[пусто]", "неклассифицированный"])].copy()
n_ipu = len(df_ipu)
print(f"Карточек на этапе ИПУ: {len(df_ipu_all):,}")
print(f"  пустой результат ИПУ: {empty_ipu:,}")
print(f"  неклассифицированный: {uncl_ipu:,}")
print(f"Для анализа результатов ИПУ: {n_ipu:,}")

print("\n" + "=" * 70)
print(f"  РЕЗУЛЬТАТЫ ИПУ ({n_ipu:,} карточек)")
print("=" * 70)

# Сводка по классам
ipu_cls = df_ipu["ipu_class"].value_counts()
ipu_cls_pct = (ipu_cls / n_ipu * 100).round(1)
ipu_summary = pd.DataFrame({"Количество": ipu_cls, "%": ipu_cls_pct})
ipu_summary.index.name = "Класс результата ИПУ"
display(ipu_summary)

# Отдельная таблица для каждой группы
for cls_name in ["положительный", "отрицательный", "активный", "ошибка"]:
    sub = df_ipu[df_ipu["ipu_class"] == cls_name]
    if len(sub) == 0:
        continue
    detail = sub["ipu"].value_counts().reset_index()
    detail.columns = ["Результат ИПУ", "Количество"]
    detail["% от группы"] = (detail["Количество"] / len(sub) * 100).round(1)
    detail["% от всех"] = (detail["Количество"] / n_ipu * 100).round(2)
    print(f"\n{'─'*60}")
    print(f"  {cls_name.upper()} ({len(sub):,} карточек, {len(sub)/n_ipu*100:.1f}%)")
    print(f"{'─'*60}")
    display(detail.style.hide(axis="index"))

### 3.1. Активные ИПУ: отдельный анализ

Активные ИПУ — карточки, по которым ИПУ ещё не завершился (даты окончания за пределами периода анализа или результат = «активный»).
Эти карточки **исключаются** из основного анализа продолжительности (секции 16-17), но показываются отдельно.

In [ ]:
print("=" * 70)
print("  АКТИВНЫЕ ИПУ: ОТДЕЛЬНЫЙ АНАЛИЗ")
print("=" * 70)

df_ipu_all_active = df[df["stream"] == "ipu"].copy()

active_by_class = df_ipu_all_active[df_ipu_all_active["ipu_class"] == "активный"]
n_active_cls = len(active_by_class)

# Дополнительно: ИПУ, у которых NEW_PLAN_END_DATE_EVT > 2025-12-31
cutoff = pd.Timestamp("2025-12-31")
has_future_date = df_ipu_all_active[
    (df_ipu_all_active["_NEW_PLAN_END_DATE_EVT_dt"] > cutoff) &
    (df_ipu_all_active["ipu_class"] != "активный")
]
n_future = len(has_future_date)

print(f"\nКарточек на этапе ИПУ (всего): {len(df_ipu_all_active):,}")
print(f"Активные по классификации результата: {n_active_cls:,}")
print(f"Дополнительно: NEW_PLAN_END_DATE > 2025-12-31, но результат != 'активный': {n_future:,}")
all_active_keys = pd.concat([
    active_by_class[DEDUP_KEY],
    has_future_date[DEDUP_KEY]
]).drop_duplicates()
n_all_active = len(all_active_keys)
print(f"Итого активных/незавершённых (объединение): {n_all_active:,}")

if n_active_cls > 0:
    print(f"\nРезультаты активных ИПУ:")
    for v, cnt in active_by_class["ipu"].value_counts().items():
        print(f"  {cnt:>4,}  «{v}»")

    print(f"\nАктивные по сегментам и типу ФП/СФП:")
    act_seg = active_by_class.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
    act_seg["Всего"] = act_seg.sum(axis=1)
    display(act_seg)

if n_future > 0:
    print(f"\nКарточки с будущей датой ИПУ (результат != 'активный'):")
    for cls, cnt in has_future_date["ipu_class"].value_counts().items():
        print(f"  {cls:25s} {cnt:,}")

print(f"\n→ Активные ИПУ ({n_all_active:,} карточек) исключены из анализа "
      "продолжительности в секциях 15-17.")

## 4. Переход от сценария к ИПУ

In [ ]:
print("=" * 70)
print("  ПЕРЕХОД ОТ СЦЕНАРИЯ К ИПУ")
print("=" * 70)

total = len(df)
to_ipu = (df["stream"] == "ipu").sum()
only_scr = total - to_ipu

print(f"\nВсего карточек:               {total:,}")
print(f"Закрыто на этапе сценария:    {only_scr:,} ({only_scr/total*100:.1f}%)")
print(f"Перешло на ИПУ:               {to_ipu:,} ({to_ipu/total*100:.1f}%)")

# Разбивка по ФП / СФП
print("\nПо типу ФП/СФП:")
trans_by_type = df.groupby("fp_type")["stream"].value_counts().unstack(fill_value=0)
trans_by_type["Всего"] = trans_by_type.sum(axis=1)
if "ipu" in trans_by_type.columns:
    trans_by_type["% на ИПУ"] = (trans_by_type["ipu"] / trans_by_type["Всего"] * 100).round(1)
display(trans_by_type)

## 5. Закрытие на стадии сценария: ФП vs СФП

In [ ]:
print("=" * 70)
print("  ЗАКРЫТИЕ НА СТАДИИ СЦЕНАРИЯ ПО ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df[df["fp_type"] == fp_t]
    n = len(sub)
    if n == 0:
        continue

    pos = (sub["scr_class"] == "положительный").sum()
    neg = (sub["scr_class"] == "отрицательный").sum()
    cpos = (sub["scr_class"] == "условно положительный").sum()
    neg = (sub["scr_class"] == "отрицательный").sum()
    cneg = (sub["scr_class"] == "условно отрицательный").sum()
    err = (sub["scr_class"] == "ошибка").sum()
    act = (sub["scr_class"] == "активный").sum()
    to_ipu_n = (sub["stream"] == "ipu").sum()

    print(f"\n--- {fp_t} ({n:,} карточек) ---")
    rows = [
        ["Положительный", pos, f"{pos/n*100:.1f}%"],
        ["Усл. положительный", cpos, f"{cpos/n*100:.1f}%"],
        ["Отрицательный", neg, f"{neg/n*100:.1f}%"],
        ["Усл. отрицательный", cneg, f"{cneg/n*100:.1f}%"],
        ["Ошибка", err, f"{err/n*100:.1f}%"],
        ["Активный", act, f"{act/n*100:.1f}%"],
        ["Перешло на ИПУ", to_ipu_n, f"{to_ipu_n/n*100:.1f}%"],
    ]
    tbl = pd.DataFrame(rows, columns=["Результат сценария", "Количество", "%"])
    display(tbl.style.hide(axis="index"))

## 6. Результаты сценария по сегментам

In [ ]:
print("=" * 70)
print("  РЕЗУЛЬТАТЫ СЦЕНАРИЯ ПО СЕГМЕНТАМ И ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df[df["fp_type"] == fp_t]
    if len(sub) == 0:
        continue

    print(f"\n{'='*40}")
    print(f"  {fp_t}")
    print(f"{'='*40}")

    # Абсолютные
    ct = pd.crosstab(sub["segment"], sub["scr_class"], margins=True, margins_name="Итого")
    print("\nАбсолютные значения:")
    display(ct)

    # Проценты внутри сегмента
    ct_pct = pd.crosstab(sub["segment"], sub["scr_class"], normalize="index") * 100
    ct_pct = ct_pct.round(1)
    print("\nПроценты (внутри сегмента):")
    display(ct_pct)

    # Переход на ИПУ по сегментам
    ipu_seg = sub.groupby("segment").apply(
        lambda g: pd.Series({
            "Всего": len(g),
            "На ИПУ": (g["stream"] == "ipu").sum(),
        })
    ).astype(int)
    ipu_seg["% на ИПУ"] = (ipu_seg["На ИПУ"] / ipu_seg["Всего"] * 100).round(1)
    print("\nПереход на ИПУ:")
    display(ipu_seg)

## 7. Закрытие на стадии ИПУ: ФП vs СФП

In [ ]:
print("=" * 70)
print("  ЗАКРЫТИЕ НА СТАДИИ ИПУ ПО ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df_ipu[df_ipu["fp_type"] == fp_t]
    n = len(sub)
    if n == 0:
        print(f"\n--- {fp_t}: нет карточек на этапе ИПУ ---")
        continue

    pos  = (sub["ipu_class"] == "положительный").sum()
    neg  = (sub["ipu_class"] == "отрицательный").sum()
    act  = (sub["ipu_class"] == "активный").sum()
    err  = (sub["ipu_class"] == "ошибка").sum()

    print(f"\n--- {fp_t} ({n:,} карточек на ИПУ) ---")
    rows = [
        ["Положительный", pos, f"{pos/n*100:.1f}%"],
        ["Отрицательный", neg, f"{neg/n*100:.1f}%"],
        ["Активный", act, f"{act/n*100:.1f}%"],
        ["Ошибка", err, f"{err/n*100:.1f}%"],
    ]
    tbl = pd.DataFrame(rows, columns=["Результат ИПУ", "Количество", "%"])
    display(tbl.style.hide(axis="index"))

## 8. Результаты ИПУ по сегментам

In [ ]:
print("=" * 70)
print("  РЕЗУЛЬТАТЫ ИПУ ПО СЕГМЕНТАМ И ТИПУ ФП/СФП")
print("=" * 70)

for fp_t in ["ФП", "СФП"]:
    sub = df_ipu[df_ipu["fp_type"] == fp_t]
    if len(sub) == 0:
        print(f"\n--- {fp_t}: нет карточек на этапе ИПУ ---")
        continue

    print(f"\n{'='*40}")
    print(f"  {fp_t}")
    print(f"{'='*40}")

    ct = pd.crosstab(sub["segment"], sub["ipu_class"], margins=True, margins_name="Итого")
    print("\nАбсолютные значения:")
    display(ct)

    ct_pct = pd.crosstab(sub["segment"], sub["ipu_class"], normalize="index") * 100
    ct_pct = ct_pct.round(1)
    print("\nПроценты (внутри сегмента):")
    display(ct_pct)

## 9. Статистика по номерам ФП

In [ ]:
print("=" * 70)
print("  СТАТИСТИКА ПО НОМЕРАМ ФП (результаты сценариев)")
print("=" * 70)

fp_only = df[df["fp_type"] == "ФП"].copy()

fp_stats = fp_only.groupby("fp_num").apply(lambda g: pd.Series({
    "Всего": len(g),
    "Положительных": (g["scr_class"] == "положительный").sum(),
    "Отрицательных": (g["scr_class"] == "отрицательный").sum(),
    "На ИПУ": (g["stream"] == "ipu").sum(),
    "Усл.полож.": (g["scr_class"] == "условно положительный").sum(),
    "Усл.отриц.": (g["scr_class"] == "условно отрицательный").sum(),
    "Активных": (g["scr_class"] == "активный").sum(),
    "Ошибок": (g["scr_class"] == "ошибка").sum(),
})).astype(int).reset_index()

fp_stats["% полож."] = (fp_stats["Положительных"] / fp_stats["Всего"] * 100).round(1)
fp_stats["% отриц."] = (fp_stats["Отрицательных"] / fp_stats["Всего"] * 100).round(1)
fp_stats["% на ИПУ"] = (fp_stats["На ИПУ"] / fp_stats["Всего"] * 100).round(1)

fp_stats = fp_stats.sort_values("Всего", ascending=False)
display(
    fp_stats[
        ["fp_num", "Всего", "Положительных", "% полож.",
         "Отрицательных", "% отриц.", "На ИПУ", "% на ИПУ",
         "Усл.полож.", "Усл.отриц.", "Активных", "Ошибок"]
    ].style.hide(axis="index").format({
        "% полож.": "{:.1f}", "% отриц.": "{:.1f}", "% на ИПУ": "{:.1f}"
    })
)

## 10. Статистика по номерам СФП

In [ ]:
print("=" * 70)
print("  СТАТИСТИКА ПО НОМЕРАМ СФП (результаты сценариев)")
print("=" * 70)

sfp_only = df[df["fp_type"] == "СФП"].copy()

if len(sfp_only) == 0:
    print("Нет карточек СФП в датасете")
else:
    sfp_stats = sfp_only.groupby("fp_num").apply(lambda g: pd.Series({
        "Всего": len(g),
        "Положительных": (g["scr_class"] == "положительный").sum(),
        "Отрицательных": (g["scr_class"] == "отрицательный").sum(),
        "На ИПУ": (g["stream"] == "ipu").sum(),
        "Усл.полож.": (g["scr_class"] == "условно положительный").sum(),
    "Усл.отриц.": (g["scr_class"] == "условно отрицательный").sum(),
    "Активных": (g["scr_class"] == "активный").sum(),
        "Ошибок": (g["scr_class"] == "ошибка").sum(),
    })).astype(int).reset_index()

    sfp_stats["% полож."] = (sfp_stats["Положительных"] / sfp_stats["Всего"] * 100).round(1)
    sfp_stats["% отриц."] = (sfp_stats["Отрицательных"] / sfp_stats["Всего"] * 100).round(1)
    sfp_stats["% на ИПУ"] = (sfp_stats["На ИПУ"] / sfp_stats["Всего"] * 100).round(1)

    sfp_stats = sfp_stats.sort_values("Всего", ascending=False)
    display(
        sfp_stats[
            ["fp_num", "Всего", "Положительных", "% полож.",
             "Отрицательных", "% отриц.", "На ИПУ", "% на ИПУ",
             "Усл.полож.", "Усл.отриц.", "Активных", "Ошибок"]
        ].style.hide(axis="index").format({
            "% полож.": "{:.1f}", "% отриц.": "{:.1f}", "% на ИПУ": "{:.1f}"
        })
    )

## 11. Статистика по номерам ФП/СФП на стадии ИПУ

In [ ]:
print("=" * 70)
print("  СТАТИСТИКА ПО НОМЕРАМ ФП/СФП (результаты ИПУ)")
print("=" * 70)

if len(df_ipu) == 0:
    print("Нет карточек на этапе ИПУ")
else:
    for fp_t in ["ФП", "СФП"]:
        sub = df_ipu[df_ipu["fp_type"] == fp_t]
        if len(sub) == 0:
            print(f"\n--- {fp_t}: нет карточек на ИПУ ---")
            continue

        print(f"\n{'='*40}")
        print(f"  {fp_t}")
        print(f"{'='*40}")

        ipu_stats = sub.groupby("fp_num").apply(lambda g: pd.Series({
            "Всего": len(g),
            "Положительных": (g["ipu_class"] == "положительный").sum(),
            "Отрицательных": (g["ipu_class"] == "отрицательный").sum(),
            "Активных": (g["ipu_class"] == "активный").sum(),
            "Ошибок": (g["ipu_class"] == "ошибка").sum(),
        })).astype(int).reset_index()

        ipu_stats["% полож."] = (ipu_stats["Положительных"] / ipu_stats["Всего"] * 100).round(1)
        ipu_stats["% отриц."] = (ipu_stats["Отрицательных"] / ipu_stats["Всего"] * 100).round(1)
        ipu_stats["% актив."] = (ipu_stats["Активных"] / ipu_stats["Всего"] * 100).round(1)

        ipu_stats = ipu_stats.sort_values("Всего", ascending=False)
        display(
            ipu_stats[
                ["fp_num", "Всего", "Положительных", "% полож.",
                 "Отрицательных", "% отриц.",
                 "Активных", "% актив.",
                 "Ошибок"]
            ].style.hide(axis="index").format({
                "% полож.": "{:.1f}", "% отриц.": "{:.1f}", "% актив.": "{:.1f}"
            })
        )

## 12. Примеры записей ИПУ: «ошибка» и «активные»

Выборка записей из дедуплицированного датасета для каждого результата из групп «ошибка» и «активный».

In [ ]:
SHOW_COLS = ["inn", "fp_num", "fp_type", "segment", "IDENTIFICATION_DATE",
             "scenario", "ipu",
             "END_DATE_SCR_FCT", "END_DATE_SCR_PLAN",
             "FIRST_END_DATE_EVENT", "END_EVENT_DATE_FACT", "NEW_PLAN_END_DATE_EVT",
             "DATE_END_FP_SFP"]
show_cols = [c for c in SHOW_COLS if c in df_ipu.columns]

for cls_name, cls_label in [("ошибка", "ОШИБКА"), ("активный", "АКТИВНЫЙ")]:
    sub = df_ipu[df_ipu["ipu_class"] == cls_name]
    if len(sub) == 0:
        print(f"\n--- {cls_label}: нет карточек ---")
        continue

    for result_val in sub["ipu"].unique():
        sample = sub[sub["ipu"] == result_val]
        n_show = min(5, len(sample))
        print(f"\n{'─'*70}")
        print(f"  {cls_label}: «{result_val}»  ({len(sample):,} карточек, показано {n_show})")
        print(f"{'─'*70}")
        display(sample[show_cols].head(n_show).style.hide(axis="index"))

## 13. Техническое закрытие ФП с открытием нового сценария

Как часто уникальные карточки ФП/СФП (по `DEDUP_KEY`) имели хотя бы одну запись в `raw` с результатом сценария «Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария» (и сегментные варианты).

In [ ]:
TECH_CLOSE_SCENARIOS = [norm_text(v) for v in [
    "Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "7_ККБ/МСБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "Микро_У Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
    "ДКБ_Техническое закрытие ФП, открытие нового ФП в связи с запуском нового сценария",
]]

raw_with_seg = raw.merge(
    df[DEDUP_KEY + ["segment"]].drop_duplicates(DEDUP_KEY),
    on=DEDUP_KEY, how="inner", suffixes=("", "_df")
)

has_tech = raw_with_seg[raw_with_seg["scenario"].isin(TECH_CLOSE_SCENARIOS)]
cards_with_tech = has_tech.drop_duplicates(DEDUP_KEY)

n_df = len(df)
n_tech = len(cards_with_tech)
print(f"Карточек с тех. закрытием (хотя бы 1 запись): {n_tech:,} из {n_df:,} ({n_tech/n_df*100:.1f}%)")

print("\nПо результату:")
for v in TECH_CLOSE_SCENARIOS:
    hits = has_tech[has_tech["scenario"] == v].drop_duplicates(DEDUP_KEY)
    print(f"  {len(hits):>5,}  {v}")

print("\nПо сегментам и типу ФП/СФП:")
tech_seg = cards_with_tech.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
tech_seg["Всего"] = tech_seg.sum(axis=1)
display(tech_seg)

total_seg = df.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
pct_seg = (tech_seg / total_seg * 100).fillna(0).round(1)
pct_seg = pct_seg.drop(columns=["Всего"], errors="ignore")
print("\n% карточек с тех. закрытием (от общего числа в сегменте):")
display(pct_seg)

### 13.2. Смена сценариев: аналитика

Определяем, сколько уникальных карточек ФП/СФП имели **смену сценария** — т.е. в `raw` у них присутствуют строки с **разными** результатами сценария (`DESC_TEXT_1`). Смена сценария определяется по наличию >1 уникального ненулевого результата для одного `DEDUP_KEY`.

In [ ]:
print("=" * 70)
print("  СМЕНА СЦЕНАРИЕВ: АНАЛИТИКА")
print("=" * 70)

raw_nonempty = raw[raw["scenario"] != ""].copy()
scr_per_card = raw_nonempty.groupby(DEDUP_KEY)["scenario"].nunique().reset_index()
scr_per_card.columns = DEDUP_KEY + ["n_unique_results"]

cards_with_change = scr_per_card[scr_per_card["n_unique_results"] > 1]
total_cards = len(scr_per_card)
n_changed = len(cards_with_change)

print(f"\nВсего карточек с >=1 результатом сценария: {total_cards:,}")
print(f"Карточек с >1 уникальным результатом (смена сценария): "
      f"{n_changed:,} ({n_changed/max(total_cards,1)*100:.1f}%)")

if n_changed > 0:
    changed_keys = cards_with_change[DEDUP_KEY]
    changed_final = df.merge(changed_keys, on=DEDUP_KEY, how="inner")

    print(f"\nКонечный результат карточек со сменой сценария:")
    final_dist = changed_final["scr_class"].value_counts()
    for cls, cnt in final_dist.items():
        print(f"  {cls:30s} {cnt:>5,} ({cnt/n_changed*100:.1f}%)")

    print(f"\nРазбивка по количеству смен и конечному результату:")
    merged = changed_final.merge(cards_with_change[DEDUP_KEY + ["n_unique_results"]],
                                  on=DEDUP_KEY, how="left")
    ct = pd.crosstab(merged["n_unique_results"], merged["scr_class"],
                     margins=True, margins_name="Итого")
    display(ct)

    print(f"\nПо сегментам и типу ФП/СФП:")
    seg_change = changed_final.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
    seg_total = df.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
    seg_pct = (seg_change / seg_total * 100).fillna(0).round(1)
    print("\nАбсолютные:")
    display(seg_change)
    print("\n% от общего числа карточек в сегменте:")
    display(seg_pct)

    print(f"\nПо потокам (сценарий vs ИПУ):")
    stream_dist = changed_final["stream"].value_counts()
    for s, cnt in stream_dist.items():
        print(f"  {s:15s} {cnt:>5,}")
else:
    print("\nНе обнаружено карточек со сменой сценария.")

## 14. ИПУ: «Требует корректировки ИПУ»

Как часто уникальные карточки ФП/СФП имели хотя бы одну запись в `raw` с результатом ИПУ:  
*«ФП не устранен в рамках Плана, оказывает негативное влияние... (требует корректировки ИПУ)»*

In [ ]:
IPU_KORR = norm_text(
    "ФП не устранен в рамках Плана, оказывает негативное влияние на исполнение "
    "участником кредитной сделки обязательств перед Банком (требует корректировки ИПУ)"
)

has_korr = raw_with_seg[raw_with_seg["ipu"] == IPU_KORR]
cards_korr = has_korr.drop_duplicates(DEDUP_KEY)

n_ipu_all = (df["stream"] == "ipu").sum()
n_korr = len(cards_korr)
print(f"Карточек с «требует корректировки ИПУ» (хотя бы 1 запись): "
      f"{n_korr:,} из {n_ipu_all:,} карточек на ИПУ ({n_korr/max(n_ipu_all,1)*100:.1f}%)")

print("\nПо сегментам и типу ФП/СФП:")
korr_seg = cards_korr.groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
korr_seg["Всего"] = korr_seg.sum(axis=1)
display(korr_seg)

ipu_total_seg = df[df["stream"] == "ipu"].groupby(["segment", "fp_type"]).size().unstack(fill_value=0)
pct_korr = (korr_seg / ipu_total_seg * 100).fillna(0).round(1)
pct_korr = pct_korr.drop(columns=["Всего"], errors="ignore")
print("\n% от карточек на ИПУ в сегменте:")
display(pct_korr)

## 15. Расхождение план/факт дат сценариев и ИПУ

- **Сценарий:** `END_DATE_SCR_PLAN` vs `END_DATE_SCR_FCT` (дней)
- **ИПУ:** `FIRST_END_DATE_EVENT` vs `END_EVENT_DATE_FACT` (дней)

Положительное значение = факт позже плана (просрочка), отрицательное = факт раньше плана.

In [ ]:
print("=" * 70)
print("  РАСХОЖДЕНИЕ ПЛАН / ФАКТ (дней)")
print("=" * 70)

# Сценарий: факт - план
df["_scr_plan"] = df["_END_DATE_SCR_PLAN_dt"]
df["_scr_fact"] = df["_END_DATE_SCR_FCT_dt"]
df["scr_diff_days"] = (df["_scr_fact"] - df["_scr_plan"]).dt.days

scr_diff = df.dropna(subset=["scr_diff_days"])
print(f"\nСценарий: карточек с обеими датами: {len(scr_diff):,}")

_abs = scr_diff["scr_diff_days"].abs()
p95_scr = np.percentile(_abs.dropna(), 95)
scr_main = scr_diff[_abs <= p95_scr]
scr_outliers = scr_diff[_abs > p95_scr]
print(f"95-й перцентиль (|расхождение|): {p95_scr:.0f} дней")
print(f"Основная масса (<= P95): {len(scr_main):,} карточек")
print(f"Выбросы (> P95, вероятные ошибки в датах): {len(scr_outliers):,} "
      f"({len(scr_outliers)/max(len(scr_diff),1)*100:.1f}%)")
if len(scr_outliers) > 0:
    print(f"  Диапазон выбросов: от {scr_outliers['scr_diff_days'].min():.0f} до "
          f"{scr_outliers['scr_diff_days'].max():.0f} дней")

scr_agg = scr_main.groupby(["segment", "fp_type"])["scr_diff_days"].agg(
    ["count", "mean", "median", "min", "max", "std"]
).round(1)
scr_agg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
print("\nРасхождение план/факт сценария по сегментам (без выбросов, P95):")
display(scr_agg)

scr_by_fp = scr_main.groupby("fp_num")["scr_diff_days"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1).reset_index()
scr_by_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
scr_by_fp = scr_by_fp.sort_values("Кол-во", ascending=False)
print("\nПо номерам ФП/СФП (без выбросов):")
display(scr_by_fp.style.hide(axis="index").format({
    "Среднее (дн)": "{:.1f}", "Медиана (дн)": "{:.0f}", "Мин": "{:.0f}", "Макс": "{:.0f}"
}))

# ИПУ: факт - план
df["_ipu_plan"] = df["_FIRST_END_DATE_EVENT_dt"]
df["_ipu_fact"] = df["_END_EVENT_DATE_FACT_dt"]
df["ipu_diff_days"] = (df["_ipu_fact"] - df["_ipu_plan"]).dt.days

ipu_diff = df[df["stream"] == "ipu"].dropna(subset=["ipu_diff_days"])
ipu_diff = ipu_diff[~ipu_diff["ipu_class"].isin(["активный"])].copy()
print(f"\n\nИПУ: карточек с обеими датами (без активных): {len(ipu_diff):,}")

if len(ipu_diff) > 0:
    _abs_i = ipu_diff["ipu_diff_days"].abs()
    p95_ipu = np.percentile(_abs_i.dropna(), 95)
    ipu_main = ipu_diff[_abs_i <= p95_ipu]
    ipu_outliers = ipu_diff[_abs_i > p95_ipu]
    print(f"95-й перцентиль (|расхождение|): {p95_ipu:.0f} дней")
    print(f"Основная масса (<= P95): {len(ipu_main):,}")
    print(f"Выбросы (> P95): {len(ipu_outliers):,} "
          f"({len(ipu_outliers)/max(len(ipu_diff),1)*100:.1f}%)")
    if len(ipu_outliers) > 0:
        print(f"  Диапазон выбросов: от {ipu_outliers['ipu_diff_days'].min():.0f} до "
              f"{ipu_outliers['ipu_diff_days'].max():.0f} дней")

    ipu_agg = ipu_main.groupby(["segment", "fp_type"])["ipu_diff_days"].agg(
        ["count", "mean", "median", "min", "max", "std"]
    ).round(1)
    ipu_agg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
    print("\nРасхождение план/факт ИПУ по сегментам (без выбросов, P95):")
    display(ipu_agg)

    ipu_by_fp = ipu_main.groupby("fp_num")["ipu_diff_days"].agg(
        ["count", "mean", "median", "min", "max"]
    ).round(1).reset_index()
    ipu_by_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
    ipu_by_fp = ipu_by_fp.sort_values("Кол-во", ascending=False)
    print("\nПо номерам ФП/СФП (без выбросов):")
    display(ipu_by_fp.style.hide(axis="index").format({
        "Среднее (дн)": "{:.1f}", "Медиана (дн)": "{:.0f}", "Мин": "{:.0f}", "Макс": "{:.0f}"
    }))
else:
    print("Недостаточно данных для анализа расхождения ИПУ")

## 16. Время от выявления ФП до закрытия (только сценарий)

Для карточек, которые **не переходят на ИПУ** (поток 1): время от `IDENTIFICATION_DATE` до `DATE_END_FP_SFP` (дней).

In [ ]:
print("=" * 70)
print("  ВРЕМЯ ОТ ВЫЯВЛЕНИЯ ДО ЗАКРЫТИЯ: ТОЛЬКО СЦЕНАРИЙ (поток 1)")
print("=" * 70)

scr_only = df[df["stream"] == "scenario"].copy()
scr_only["_close_dt"] = scr_only["_DATE_END_FP_SFP_dt"]
scr_only["days_to_close"] = (scr_only["_close_dt"] - scr_only["dt"]).dt.days

scr_close = scr_only.dropna(subset=["days_to_close"])
scr_close = scr_close[scr_close["days_to_close"] >= 0]
print(f"\nКарточек с обеими датами (>= 0 дней): {len(scr_close):,} из {len(scr_only):,}")

p95_sc = np.percentile(scr_close["days_to_close"].dropna(), 95)
scr_close_main = scr_close[scr_close["days_to_close"] <= p95_sc]
scr_close_out = scr_close[scr_close["days_to_close"] > p95_sc]
print(f"95-й перцентиль: {p95_sc:.0f} дней")
print(f"Основная масса (<= P95): {len(scr_close_main):,}")
print(f"Выбросы (> P95, вероятные ошибки в датах): {len(scr_close_out):,} "
      f"({len(scr_close_out)/max(len(scr_close),1)*100:.1f}%)")
if len(scr_close_out) > 0:
    print(f"  Диапазон выбросов: от {scr_close_out['days_to_close'].min():.0f} до "
          f"{scr_close_out['days_to_close'].max():.0f} дней")

agg_seg = scr_close_main.groupby(["segment", "fp_type"])["days_to_close"].agg(
    ["count", "mean", "median", "min", "max", "std"]
).round(1)
agg_seg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
print("\nПо сегментам (без выбросов, P95):")
display(agg_seg)

agg_fp = scr_close_main.groupby("fp_num")["days_to_close"].agg(
    ["count", "mean", "median", "min", "max"]
).round(1).reset_index()
agg_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
agg_fp = agg_fp.sort_values("Кол-во", ascending=False)
print("\nПо номерам ФП/СФП (без выбросов):")
display(agg_fp.style.hide(axis="index").format({
    "Среднее (дн)": "{:.1f}", "Медиана (дн)": "{:.0f}", "Мин": "{:.0f}", "Макс": "{:.0f}"
}))

## 17. Время от выявления ФП до закрытия (с переходом на ИПУ)

Для карточек, которые **переходят на ИПУ** (поток 2): время от `IDENTIFICATION_DATE` до `DATE_END_FP_SFP` (дней).

In [ ]:
print("=" * 70)
print("  ВРЕМЯ ОТ ВЫЯВЛЕНИЯ ДО ЗАКРЫТИЯ: С ПЕРЕХОДОМ НА ИПУ (поток 2)")
print("=" * 70)

ipu_stream = df[df["stream"] == "ipu"].copy()
ipu_stream = ipu_stream[~ipu_stream["ipu_class"].isin(["активный"])].copy()
ipu_stream["_close_dt"] = ipu_stream["_DATE_END_FP_SFP_dt"]
ipu_stream["days_to_close"] = (ipu_stream["_close_dt"] - ipu_stream["dt"]).dt.days

ipu_close = ipu_stream.dropna(subset=["days_to_close"])
ipu_close = ipu_close[ipu_close["days_to_close"] >= 0]
print(f"\nКарточек с обеими датами (>= 0 дней, без активных ИПУ): "
      f"{len(ipu_close):,} из {len(ipu_stream):,}")

if len(ipu_close) > 0:
    p95_ic = np.percentile(ipu_close["days_to_close"].dropna(), 95)
    ipu_close_main = ipu_close[ipu_close["days_to_close"] <= p95_ic]
    ipu_close_out = ipu_close[ipu_close["days_to_close"] > p95_ic]
    print(f"95-й перцентиль: {p95_ic:.0f} дней")
    print(f"Основная масса (<= P95): {len(ipu_close_main):,}")
    print(f"Выбросы (> P95, вероятные ошибки в датах): {len(ipu_close_out):,} "
          f"({len(ipu_close_out)/max(len(ipu_close),1)*100:.1f}%)")
    if len(ipu_close_out) > 0:
        print(f"  Диапазон выбросов: от {ipu_close_out['days_to_close'].min():.0f} до "
              f"{ipu_close_out['days_to_close'].max():.0f} дней")

    agg_seg = ipu_close_main.groupby(["segment", "fp_type"])["days_to_close"].agg(
        ["count", "mean", "median", "min", "max", "std"]
    ).round(1)
    agg_seg.columns = ["Кол-во", "Среднее", "Медиана", "Мин", "Макс", "Стд"]
    print("\nПо сегментам (без выбросов, P95):")
    display(agg_seg)

    agg_fp = ipu_close_main.groupby("fp_num")["days_to_close"].agg(
        ["count", "mean", "median", "min", "max"]
    ).round(1).reset_index()
    agg_fp.columns = ["fp_num", "Кол-во", "Среднее (дн)", "Медиана (дн)", "Мин", "Макс"]
    agg_fp = agg_fp.sort_values("Кол-во", ascending=False)
    print("\nПо номерам ФП/СФП (без выбросов):")
    display(agg_fp.style.hide(axis="index").format({
        "Среднее (дн)": "{:.1f}", "Медиана (дн)": "{:.0f}", "Мин": "{:.0f}", "Макс": "{:.0f}"
    }))

# Сравнение двух потоков (P95)
print("\n" + "=" * 70)
print("  СРАВНЕНИЕ: ТОЛЬКО СЦЕНАРИЙ vs С ИПУ (в пределах P95)")
print("=" * 70)
comp = pd.DataFrame({
    "Только сценарий": scr_close_main["days_to_close"].describe(),
    "С переходом на ИПУ": ipu_close_main["days_to_close"].describe() if len(ipu_close) > 0 else pd.Series(dtype=float),
}).round(1)
display(comp)

In [ ]:
doc = Document()
for sec in doc.sections:
    sec.top_margin = Cm(1.5); sec.bottom_margin = Cm(1)
    sec.left_margin = Cm(1.5); sec.right_margin = Cm(1)
style = doc.styles["Normal"]
style.font.name = "Calibri"; style.font.size = Pt(10); style.font.color.rgb = RGBColor(0, 0, 0)
for hn in ["Heading 1", "Heading 2"]:
    if hn in doc.styles: doc.styles[hn].font.color.rgb = RGBColor(0, 0, 0)

# --- Титульная ---
p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("Отчёт по EDA:\nрезультаты сценариев урегулирования и ИПУ")
r.font.size = Pt(20); r.font.bold = True; r.font.color.rgb = RGBColor(0, 0, 0)
p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
r = p.add_run("Сегменты: МкБ · МСБ · ККБ"); r.font.size = Pt(11); r.font.italic = True
doc.add_page_break()
_heading(doc, "Оглавление", level=1); _add_toc(doc); doc.add_page_break()

# Подготовка данных
df_scr = df[df["stream"] == "scenario"].copy()
df_scr_clean = df_scr[~df_scr["scr_class"].isin(["[пусто]", "неклассифицированный"])]
df_ipu_full = df[df["stream"] == "ipu"].copy()
df_ipu_c = df_ipu_full[~df_ipu_full["ipu_class"].isin(["[пусто]", "неклассифицированный"])]

n_total_cards = len(df.drop_duplicates(subset=DEDUP_KEY))
n_scr = len(df_scr_clean)
n_ipu_c = len(df_ipu_c)

# ═══ 1. Общая статистика ═══
_heading(doc, "1. Общая статистика по жизненному циклу ФП/СФП", level=1)
_para(doc, f"Всего карточек в датасете: {_fmt(len(df)):}. "
      f"Из них на стадии сценария: {_fmt(n_scr)}, на стадии ИПУ: {_fmt(n_ipu_c)}.")

rows_ov = [
    ("Карточек (сценарий)", _fmt(n_scr)),
    ("Карточек (ИПУ)", _fmt(n_ipu_c)),
    ("Сегмент МкБ", _fmt(len(df[df["segment"]=="МкБ"]))),
    ("Сегмент МСБ", _fmt(len(df[df["segment"]=="МСБ"]))),
    ("Сегмент ККБ", _fmt(len(df[df["segment"]=="ККБ"]))),
]
_mktable(doc, ["Показатель", "Значение"], rows_ov, num_cols={1}, col_widths=[8, 5])

# ═══ 2. Результаты сценариев ═══
_heading(doc, "2. Распределение результатов сценариев", level=1)
scr_cls = df_scr_clean["scr_class"].value_counts()
rows_s = [(cls, _fmt(cnt), f"{cnt/n_scr*100:.1f}%") for cls, cnt in scr_cls.items()]
_mktable(doc, ["Класс результата", "Количество", "%"], rows_s, num_cols={1,2}, col_widths=[6,3,3])

# ═══ 3. Результаты ИПУ ═══
_heading(doc, "3. Распределение результатов ИПУ", level=1)
ipu_cls = df_ipu_c["ipu_class"].value_counts()
rows_i = [(cls, _fmt(cnt), f"{cnt/n_ipu_c*100:.1f}%") for cls, cnt in ipu_cls.items()]
_mktable(doc, ["Класс результата", "Количество", "%"], rows_i, num_cols={1,2}, col_widths=[6,3,3])

# ═══ 4. Результаты сценариев по сегментам ═══
_heading(doc, "4. Результаты сценариев по сегментам", level=1)
for seg in SEG_ORDER:
    seg_scr = df_scr_clean[df_scr_clean["segment"] == seg]
    if len(seg_scr) == 0: continue
    doc.add_paragraph(f"{seg} ({_fmt(len(seg_scr))} карточек)", style="List Bullet").runs[0].bold = True
    vc = seg_scr["scr_class"].value_counts()
    rows_ = [(cls, _fmt(cnt), f"{cnt/len(seg_scr)*100:.1f}%") for cls, cnt in vc.items()]
    _mktable(doc, ["Класс", "Кол-во", "%"], rows_, num_cols={1,2}, col_widths=[6,3,3])

# ═══ 5. Результаты ИПУ по сегментам ═══
_heading(doc, "5. Результаты ИПУ по сегментам", level=1)
for seg in SEG_ORDER:
    seg_ipu = df_ipu_c[df_ipu_c["segment"] == seg]
    if len(seg_ipu) == 0: continue
    doc.add_paragraph(f"{seg} ({_fmt(len(seg_ipu))} карточек)", style="List Bullet").runs[0].bold = True
    vc = seg_ipu["ipu_class"].value_counts()
    rows_ = [(cls, _fmt(cnt), f"{cnt/len(seg_ipu)*100:.1f}%") for cls, cnt in vc.items()]
    _mktable(doc, ["Класс", "Кол-во", "%"], rows_, num_cols={1,2}, col_widths=[6,3,3])

# ═══ 6. Закрытие: ФП vs СФП ═══
_heading(doc, "6. Закрытие на стадии сценария: ФП vs СФП", level=1)
for fp_type in ["ФП", "СФП"]:
    sub = df_scr_clean[df_scr_clean["fp_type"] == fp_type]
    if len(sub) == 0: continue
    doc.add_paragraph(f"{fp_type} ({_fmt(len(sub))} карточек)", style="List Bullet").runs[0].bold = True
    vc = sub["scr_class"].value_counts()
    rows_ = [(cls, _fmt(cnt), f"{cnt/len(sub)*100:.1f}%") for cls, cnt in vc.items()]
    _mktable(doc, ["Класс", "Кол-во", "%"], rows_, num_cols={1,2}, col_widths=[6,3,3])

# ═══ 7. Время до закрытия ═══
_heading(doc, "7. Время от выявления ФП до закрытия", level=1)

if "scr_close" in dir() and len(scr_close) > 0:
    _para(doc, f"Только сценарий: {_fmt(len(scr_close))} карточек")
    scr_stats = scr_close["days_to_close"].describe().round(1)
    rows_ts = [(k, f"{v:.1f}") for k, v in scr_stats.items()]
    _mktable(doc, ["Показатель", "Значение (дни)"], rows_ts, num_cols={1}, col_widths=[6, 4])

if "ipu_close" in dir() and len(ipu_close) > 0:
    _para(doc, f"С переходом на ИПУ: {_fmt(len(ipu_close))} карточек")
    ipu_stats = ipu_close["days_to_close"].describe().round(1)
    rows_ti = [(k, f"{v:.1f}") for k, v in ipu_stats.items()]
    _mktable(doc, ["Показатель", "Значение (дни)"], rows_ti, num_cols={1}, col_widths=[6, 4])

if "comp" in dir():
    _para(doc, "Сравнение двух потоков:")
    rows_cp = [(str(idx)] + [f"{comp.loc[idx, c]:.1f}" for c in comp.columns]) for idx in comp.index]
    _mktable(doc, [""] + list(comp.columns), rows_cp, num_cols=set(range(1, len(comp.columns)+1)),
             col_widths=[4] + [4]*len(comp.columns))

# ═══ Заключение ═══
doc.add_page_break()
_heading(doc, "Заключение", level=1)
_para(doc, "По результатам анализа жизненного цикла ФП/СФП (сценарии и ИПУ) выявлены "
     "следующие основные результаты:")
conclusions = [
    "Распределение результатов сценариев и ИПУ показывает долю положительных, "
    "отрицательных, условно положительных, условно отрицательных и активных исходов по каждому сегменту.",
    "Существуют значимые различия в структуре результатов между ФП и СФП.",
    "Анализ времени закрытия показывает, что переход на ИПУ существенно увеличивает "
    "длительность процесса по сравнению с закрытием на стадии сценария.",
    "Детализация по номерам ФП/СФП позволяет выявить факторы, требующие наибольшего "
    "времени на урегулирование.",
    'Полные данные доступны в файле приложения: Приложение к отчету "Анализ результатов мониторинга".xlsx.',
]
for c in conclusions:
    doc.add_paragraph(c, style="List Bullet")

doc.save(_OUT)
print(f"Отчёт сохранён: {_OUT}")

In [ ]:
excel_path = f'{DATA_DIR}/Приложение к отчету "Анализ результатов мониторинга".xlsx'

df_scr = df[df["stream"] == "scenario"].copy()
df_scr_clean = df_scr[~df_scr["scr_class"].isin(["[пусто]", "неклассифицированный"])]
df_ipu_clean = df[df["stream"] == "ipu"].copy()
df_ipu_clean = df_ipu_clean[~df_ipu_clean["ipu_class"].isin(["[пусто]", "неклассифицированный"])]

with pd.ExcelWriter(excel_path, engine="openpyxl") as w:
    scr_cls = df_scr_clean["scr_class"].value_counts().reset_index()
    scr_cls.columns = ["Класс", "Количество"]
    scr_cls.to_excel(w, sheet_name="Результаты сценариев", index=False)

    ipu_cls = df_ipu_clean["ipu_class"].value_counts().reset_index()
    ipu_cls.columns = ["Класс", "Количество"]
    ipu_cls.to_excel(w, sheet_name="Результаты ИПУ", index=False)

    for seg in SEG_ORDER:
        scr_seg = df_scr_clean[df_scr_clean["segment"] == seg]
        if len(scr_seg) > 0:
            tbl = scr_seg["scr_class"].value_counts().reset_index()
            tbl.columns = ["Класс", "Количество"]
            tbl.to_excel(w, sheet_name=f"Сценарий ({seg})"[:31], index=False)

    for seg in SEG_ORDER:
        ipu_seg = df_ipu_clean[df_ipu_clean["segment"] == seg]
        if len(ipu_seg) > 0:
            tbl = ipu_seg["ipu_class"].value_counts().reset_index()
            tbl.columns = ["Класс", "Количество"]
            tbl.to_excel(w, sheet_name=f"ИПУ ({seg})"[:31], index=False)

    fp_scr = df_scr_clean.groupby(["fp_num", "fp_type", "scr_class"]).size().reset_index(name="Количество")
    fp_scr.to_excel(w, sheet_name="ФП по сценариям", index=False)

    fp_ipu = df_ipu_clean.groupby(["fp_num", "fp_type", "ipu_class"]).size().reset_index(name="Количество")
    fp_ipu.to_excel(w, sheet_name="ФП по ИПУ", index=False)

print(f"Сохранено: {excel_path}")